# Transformer Training Notebook
This notebook sets up and trains a FullTransformer model using the TransformerTrainer class on the Toxic Comment Classification dataset.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys, os

# If running in Colab, mount drive

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = '/content/drive/MyDrive/OMSCS/DeepLearning/main_repo'
    os.chdir(BASE_PATH)
    sys.path.append(BASE_PATH)
    print(f'Running in Colab, path set to {BASE_PATH}')
except ImportError:
    print('Running locally')

Running locally


In [3]:
# Dataset setup

import re, random
import nltk, numpy as np, pandas as pd, torch
from nltk.tokenize import word_tokenize
from torch.utils.data import Dataset, DataLoader
from collections import Counter

# Download punkt if needed
nltk.download('punkt')

# Data utility functions
from dataset.data_cleaning import clean_text, preprocess_and_tokenize, load_glove_embeddings, ToxicDataset

[nltk_data] Downloading package punkt to /Users/soultanis/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt to /Users/soultanis/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/soultanis/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
# Transformer configuration

class Config:
    EMBEDDING_DIM = 100
    MAX_LEN = 100
    BATCH_SIZE = 128
    NUM_EPOCHS = 20
    LR = 1e-4
    STEP_SIZE = 5
    GAMMA = 0.95
    GLOVE_PATH = 'dataset/word_embeddings/glove.6B/glove.6B.100d.txt'
    DATA_PATH = 'dataset/.cache/kagglehub/competitions/jigsaw-toxic-comment-classification-challenge/train.csv.zip'
    LABELS = ['toxic','severe_toxic','obscene','threat','insult','identity_hate']
    SEED = 42

In [5]:
# Load and preprocess data

cfg = Config()
df = pd.read_csv(cfg.DATA_PATH)
df, vocab = preprocess_and_tokenize(df, cfg.MAX_LEN)
embedding_matrix = load_glove_embeddings(cfg.GLOVE_PATH, vocab, cfg.EMBEDDING_DIM)
samples_per_cls = torch.tensor([df[col].sum() for col in cfg.LABELS], dtype=torch.float32)

print('Vocab size:', len(vocab))
print('Class counts:', samples_per_cls)

Vocab size: 245832
Class counts: tensor([15294.,  1595.,  8449.,   478.,  7877.,  1405.])


In [6]:
# Prepare DataLoaders
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    df['input_ids'].tolist(), df[cfg.LABELS].values.tolist(), test_size=0.2, random_state=cfg.SEED
)

train_ds = ToxicDataset(X_train, y_train)
val_ds   = ToxicDataset(X_val, y_val)
train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE)

In [7]:
# Import transformer model and trainer

from models.transformer.model import FullTransformer
from models.transformer.trainer import TransformerTrainer

In [8]:
# Initialize model and trainer

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

model = FullTransformer(
    src_vocab_size=len(vocab), 
    tgt_vocab_size=len(vocab), 
    pad_idx=vocab['<PAD>'],
    d_model=512, 
    nhead=8, 
    num_layers=6, 
    dim_feedforward=2048, 
    dropout=0.1
).to(device)

trainer = TransformerTrainer(
    model=model, 
    pad_idx=vocab['<PAD>'], 
    lr=cfg.LR, 
    step_size=cfg.STEP_SIZE,
    gamma=cfg.GAMMA, 
    device=device
)

In [ ]:
print(device)
PYTORCH_ENABLE_MPS_FALLBACK=1

# Training loop for the model
trainer.train(train_loader, val_loader, cfg.NUM_EPOCHS)


mps


NotImplementedError: The operator 'aten::_nested_tensor_from_mask_left_aligned' is not currently implemented for the MPS device. If you want this op to be considered for addition please comment on https://github.com/pytorch/pytorch/issues/141287 and mention use-case, that resulted in missing op as well as commit hash 134179474539648ba7dee1317959529fbd0e7f89. As a temporary fix, you can set the environment variable `PYTORCH_ENABLE_MPS_FALLBACK=1` to use the CPU as a fallback for this op. WARNING: this will be slower than running natively on MPS.

In [ ]:
# Save trained transformer
torch.save(model.state_dict(), 'transformer_toxic_comments.pth')